# Chapter 4 — Activation Functions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saanikakulkarni07/learn-neural-nets/blob/main/ch04-activation-functions/exercises.ipynb)

The chapter that turns our layers into a real classifier. We'll *see* why a network without nonlinear activations can't fit a curve, code **ReLU** for the hidden layers and **Softmax** for the output, learn the **numerical-stability trick**, and assemble the full forward pass: `Dense → ReLU → Dense → Softmax`.

**Goals**
1. Plot Step / Sigmoid / ReLU and feel what each does.
2. Demonstrate that linear-only layers can't fit `sin(x)`, but ReLU can.
3. Build `Activation_ReLU` and `Activation_Softmax` from scratch.
4. Understand `axis`/`keepdims` and the max-subtraction stability trick.
5. Run the full model and read off class confidences (+ astro framing).


## 0. Setup

In [ ]:
try:
    import nnfs
except ImportError:
    !pip install nnfs -q
    import nnfs
import numpy as np
import matplotlib.pyplot as plt
from nnfs.datasets import spiral_data
nnfs.init()
print('setup ok')

## 1. Concept check

1. Why can't a network made of only linear (`y=x`) neurons fit `y = sin(x)`, no matter how deep?
2. What's wrong with the step function from an optimizer's point of view?
3. In a ReLU neuron, what does the **weight** control vs. the **bias**?
4. Why does Softmax **exponentiate** before normalizing (two reasons)?
5. Why subtract the max before `np.exp`, and why doesn't it change the result?
6. After Softmax on an untrained net, why are outputs ~`[0.33, 0.33, 0.33]`?


_Your answers:_

1. 
2. 
3. 
4. 
5. 
6. 

<details><summary>Check yourself</summary>

1. A composition of linear functions is still linear — the whole network reduces to one straight line, which can't trace a curve.
2. It's not informative: output is just 0/1, so the optimizer can't tell how *close* a neuron was to flipping (input 3 and 300000 both give 1).
3. Weight = slope/sign of the active part (negative flips activate↔deactivate); bias = horizontal shift of the activation point (when the neuron turns on).
4. (a) `e^x` is always non-negative — no negative 'probabilities'; (b) it's monotonic, so it never changes which class wins.
5. `np.exp` overflows for large inputs (`exp(1000)=inf`). Subtracting the row max makes the largest value 0 and the rest negative → safe. Softmax normalizes, so subtracting a constant from all inputs leaves the output unchanged.
6. The net is random/untrained (random weights, zero biases), so each class gets ~equal share.
</details>

## 2. The activation functions, plotted

**Exercise 2.** Implement step, sigmoid, and ReLU, and plot them on `x ∈ [-5, 5]`. Notice ReLU = `y=x` clipped at 0, and sigmoid is the smooth (0,1) S-curve.

In [ ]:
x = np.linspace(-5, 5, 200)

step    = np.where(x > 0, 1, 0)
sigmoid = 1 / (1 + np.exp(-x))
relu    = np.maximum(0, x)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
ax[0].plot(x, step);    ax[0].set_title('Step: 1 if x>0 else 0')
ax[1].plot(x, sigmoid); ax[1].set_title('Sigmoid: 1/(1+e^-x)')
ax[2].plot(x, relu);    ax[2].set_title('ReLU: max(0, x)')
for a in ax:
    a.axhline(0, color='k', lw=0.5); a.axvline(0, color='k', lw=0.5); a.set_xlabel('x')
plt.tight_layout(); plt.show()

## 3. Linear layers can't fit a curve — ReLU can

This is the core argument of the chapter. We'll fake two stacked layers with **no activation** and confirm the output is a straight line in `x` (can't be `sin`). Then we'll show a single ReLU neuron already introduces a bend.

In [ ]:
x = np.linspace(0, 2*np.pi, 200).reshape(-1, 1)
target = np.sin(x)

# two 'layers' with NO activation (linear): output stays linear in x
W1 = np.random.randn(1, 8); b1 = np.random.randn(1, 8)
W2 = np.random.randn(8, 1); b2 = np.random.randn(1, 1)
h = x @ W1 + b1            # no activation
y_linear = h @ W2 + b2     # no activation

# fit a degree-1 line to (x, y_linear): residual ~0 proves it's just a line
slope, intercept = np.polyfit(x.ravel(), y_linear.ravel(), 1)
resid = np.max(np.abs(y_linear.ravel() - (slope*x.ravel() + intercept)))
print(f'linear-only output is a straight line; max deviation from a line = {resid:.2e}')
assert resid < 1e-6

plt.figure(figsize=(6,3.5))
plt.plot(x, target, 'g', label='target sin(x)')
plt.plot(x, y_linear, 'r', label='linear-only net (a line!)')
plt.legend(); plt.title('No activation -> cannot fit a curve'); plt.show()

**Exercise 3.** Show ReLU bends. For a single neuron `relu(w*x + b)`, set `w=1` and try a few biases; observe the activation point moves horizontally (more bias = activates earlier).

In [ ]:
xx = np.linspace(-3, 3, 200)
plt.figure(figsize=(6,3.5))
for b in [-1.0, 0.0, 1.0]:
    plt.plot(xx, np.maximum(0, 1.0*xx + b), label=f'relu(x + {b})')
plt.axhline(0, color='k', lw=0.5); plt.axvline(0, color='k', lw=0.5)
plt.legend(); plt.title('Bias shifts the ReLU activation point horizontally'); plt.show()
# A pair of ReLU neurons creates an 'area of effect'; many pairs approximate any curve.

## 4. Build `Activation_ReLU`

**Exercise 4.** Implement it (`np.maximum(0, inputs)`) and confirm negatives are clipped to 0.

In [ ]:
inputs = [0, 2, -1, 3.3, -2.7, 1.1, 2.2, -100]

class Activation_ReLU:
    def forward(self, inputs):
        self.output = np.maximum(0, inputs)

relu = Activation_ReLU()
relu.forward(inputs)
print(relu.output)
assert np.allclose(relu.output, [0, 2, 0, 3.3, 0, 1.1, 2.2, 0])
print('✓ negatives clipped to 0')

## 5. `axis` / `keepdims`, then Softmax from scratch

**Exercise 5a.** On the book's batch, show `np.sum` with `axis=1, keepdims=True` gives a **column vector** of per-row sums (the shape we need to normalize each sample).

In [ ]:
layer_outputs = np.array([[4.8, 1.21, 2.385],
                          [8.9, -1.81, 0.2],
                          [1.41, 1.051, 0.026]])
print('axis=None (scalar):', np.sum(layer_outputs))
print('axis=0 (down cols) :', np.sum(layer_outputs, axis=0))
print('axis=1 (across rows):', np.sum(layer_outputs, axis=1))
col = np.sum(layer_outputs, axis=1, keepdims=True)
print('axis=1, keepdims -> column vector shape', col.shape)
assert col.shape == (3, 1)
print('✓')

**Exercise 5b.** Build `Activation_Softmax` with the max-subtraction stability trick. Verify each row sums to 1, and that subtracting a constant doesn't change the output.

In [ ]:
class Activation_Softmax:
    def forward(self, inputs):
        exp_values = np.exp(inputs - np.max(inputs, axis=1, keepdims=True))
        self.output = exp_values / np.sum(exp_values, axis=1, keepdims=True)

softmax = Activation_Softmax()
softmax.forward([[1, 2, 3]])
print('softmax([1,2,3]) =', softmax.output)
assert np.isclose(softmax.output.sum(), 1.0)

# shift-invariance: subtract 3 from all -> identical output
softmax.forward([[-2, -1, 0]])
shifted = softmax.output.copy()
softmax.forward([[1, 2, 3]])
assert np.allclose(shifted, softmax.output)
print('✓ rows sum to 1, and softmax is shift-invariant (stability trick is free)')

**Exercise 5c.** Prove the overflow problem the trick prevents.

In [ ]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    print('naive exp(1000) =', np.exp(1000.0))   # inf -> would poison normalization
print('stabilized: exp(1000 - 1000) =', np.exp(1000.0 - 1000.0))
assert np.isinf(np.exp(1000.0)) and np.exp(0.0) == 1.0
print('✓ subtracting the max keeps exponents <= 0, so no overflow')

## 6. The full forward pass

Assemble `Dense(2,3) → ReLU → Dense(3,3) → Softmax` on the spiral data, exactly like the book. We reset the seed first for reproducibility. Output should be ~`[0.33, 0.33, 0.33]` per row — the net is **untrained**, so it's near-uniform (this is expected!).

In [ ]:
nnfs.init()  # reset seed to reproduce the book's numbers

class Layer_Dense:
    def __init__(self, n_inputs, n_neurons):
        self.weights = 0.01 * np.random.randn(n_inputs, n_neurons)
        self.biases  = np.zeros((1, n_neurons))
    def forward(self, inputs):
        self.output = np.dot(inputs, self.weights) + self.biases

X, y = spiral_data(samples=100, classes=3)

dense1 = Layer_Dense(2, 3);  activation1 = Activation_ReLU()
dense2 = Layer_Dense(3, 3);  activation2 = Activation_Softmax()

dense1.forward(X)
activation1.forward(dense1.output)
dense2.forward(activation1.output)
activation2.forward(dense2.output)

print(activation2.output[:5])
# every row is a valid probability distribution
assert np.allclose(activation2.output.sum(axis=1), 1.0)
# untrained -> roughly uniform across 3 classes
assert np.allclose(activation2.output[:5], 1/3, atol=1e-2)
print('✓ full forward pass: each row sums to 1, ~uniform because untrained')

**Exercise 6b.** Turn confidences into predictions with `argmax`, and note that confidence matters, not just the winning class.

In [ ]:
preds = np.argmax(activation2.output, axis=1)
print('first 10 predicted classes:', preds[:10])
print('argmax([0.22,0.6,0.18]) =', np.argmax([0.22, 0.6, 0.18]),
      '| argmax([0.32,0.36,0.32]) =', np.argmax([0.32, 0.36, 0.32]))
print('same class (1), but 60% confidence >> 36% — the confidence carries information')

## 7. 🔭 Astro bonus — class confidences for a tiny catalog

Run the same `Dense → ReLU → Dense → Softmax` stack on a fake `(N, 4)` photometric catalog and read the output as `[P(star), P(galaxy), P(QSO)]`. Then apply a **confidence cut** — the real-world move of trusting only high-confidence labels and flagging ambiguous sources.

**Exercise 7.** Build the stack for 4 features → 3 classes, forward the catalog, and count how many sources exceed a 0.5 confidence threshold.

In [ ]:
rng = np.random.RandomState(0)
N = 1000
catalog = rng.normal(size=(N, 4)).astype(np.float32)  # N sources, 4 color features

d1 = Layer_Dense(4, 16); a1 = Activation_ReLU()
d2 = Layer_Dense(16, 3); a2 = Activation_Softmax()
d1.forward(catalog); a1.forward(d1.output)
d2.forward(a1.output); a2.forward(d2.output)

probs = a2.output
labels = np.array(['star', 'galaxy', 'QSO'])
pred_idx = np.argmax(probs, axis=1)
confidence = probs.max(axis=1)

print('first 3 sources:')
for i in range(3):
    print(f'  P={np.round(probs[i],3)} -> {labels[pred_idx[i]]} ({confidence[i]:.2f})')

assert np.allclose(probs.sum(axis=1), 1.0)
high_conf = confidence > 0.5
print(f'\nconfident (>0.5) labels: {high_conf.sum()}/{N}; '
      f'flag the other {(~high_conf).sum()} for follow-up')
print('✓ (untrained net -> confidences hover near 1/3, so few clear the cut — '
      'training is what sharpens them)')

### Stretch
- The outputs are near-uniform because the net is untrained. Ch. 5 adds a **loss** to measure how wrong these confidences are; Ch. 6–10 train the weights so the distributions sharpen.
- Swap ReLU for sigmoid in the hidden layer and compare the output distribution.
- Try a real `(N_objects, N_features)` table (standardized) — the stack runs unchanged.